# OSINT Vortex Engine — Entity Extraction + Graph (Offline Demo)

This notebook is a **safe, offline** OSINT-style analysis demo.
It does not perform target reconnaissance. It uses **synthetic/public-like** text snippets to demonstrate:

- entity extraction (emails, domains, IPs, handles, URLs)
- normalization + deduplication
- relationship graph construction (snippet → entities)
- simple scoring (centrality-like) to prioritize investigation

Outputs are saved inside the notebook when executed.

In [1]:
import re
import numpy as np
import pandas as pd
from collections import defaultdict, Counter

SEED = 1337
rng = np.random.default_rng(SEED)
pd.set_option('display.max_columns', 100)

## 1) Synthetic corpus

In [2]:
snippets = [
  'Support: contact helpdesk@acme.example, or visit https://status.acme.example/incidents. Ref IP 203.0.113.10',
  'Marketing mentions @acme_support and mail press@acme.example. CDN at 198.51.100.42',
  'A leaked paste references admin@acme.example and https://login.acme.example/reset',
  'Third-party vendor: alerts@vendor.example, API https://api.vendor.example/v1, IP 203.0.113.10',
  'Community forum: user @john_doe posts link http://blog.acme.example/post/123',
]
df = pd.DataFrame({'snippet_id': range(len(snippets)), 'text': snippets})
df

,snippet_id,text
0,0,"Support: contact helpdesk@acme.example, or vis..."
1,1,Marketing mentions @acme_support and mail pres...
2,2,A leaked paste references admin@acme.example a...
3,3,"Third-party vendor: alerts@vendor.example, API..."
4,4,Community forum: user @john_doe posts link htt...


## 2) Entity extraction

In [3]:
RE_EMAIL = re.compile(r'\b[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}\b')
RE_URL = re.compile(r'https?://[^\s]+')
RE_IP = re.compile(r'\b(?:\d{1,3}\.){3}\d{1,3}\b')
RE_HANDLE = re.compile(r'(?<!\w)@[a-zA-Z0-9_]{3,20}')
RE_DOMAIN = re.compile(r'\b[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}\b')

def normalize_domain(d: str) -> str:
  return d.lower().strip('.')

def extract(text: str):
  emails = [m.group(0).lower() for m in RE_EMAIL.finditer(text)]
  urls = [m.group(0) for m in RE_URL.finditer(text)]
  ips = [m.group(0) for m in RE_IP.finditer(text)]
  handles = [m.group(0).lower() for m in RE_HANDLE.finditer(text)]

  domains = []
  for u in urls:
    dom = re.sub(r'^https?://', '', u).split('/')[0]
    domains.append(normalize_domain(dom))
  for e in emails:
    domains.append(normalize_domain(e.split('@')[-1]))

  # extra domain hits from free text
  for m in RE_DOMAIN.finditer(text):
    domains.append(normalize_domain(m.group(0)))

  return {
    'email': sorted(set(emails)),
    'url': sorted(set(urls)),
    'ip': sorted(set(ips)),
    'handle': sorted(set(handles)),
    'domain': sorted(set(domains)),
  }

entities = []
for r in df.itertuples(index=False):
  ent = extract(r.text)
  entities.append(ent)
entities[0]

{'email': ['helpdesk@acme.example'],
 'url': ['https://status.acme.example/incidents.'],
 'ip': ['203.0.113.10'],
 'handle': [],
 'domain': ['acme.example', 'status.acme.example']}

## 3) Build graph (snippet ↔ entity)

In [4]:
edges = []
for sid, ent in enumerate(entities):
  for kind, vals in ent.items():
    for v in vals:
      edges.append((sid, f'{kind}:{v}'))

E = pd.DataFrame(edges, columns=['snippet_id','entity'])
E.head(), E['entity'].nunique()

(   snippet_id                                      entity
 0           0                 email:helpdesk@acme.example
 1           0  url:https://status.acme.example/incidents.
 2           0                             ip:203.0.113.10
 3           0                         domain:acme.example
 4           0                  domain:status.acme.example,
 18)

## 4) Scoring / prioritization

In [5]:
deg = E.groupby('entity')['snippet_id'].nunique().sort_values(ascending=False)
top = deg.head(15).reset_index().rename(columns={'snippet_id':'support'})
top

,entity,support
0,domain:acme.example,3
1,ip:203.0.113.10,2
2,domain:api.vendor.example,1
3,domain:blog.acme.example,1
4,domain:status.acme.example,1
5,domain:login.acme.example,1
6,email:admin@acme.example,1
7,email:alerts@vendor.example,1
8,email:helpdesk@acme.example,1
9,domain:vendor.example,1


In [6]:
# show which snippets mention the top entities
out = []
for ent in top['entity'].head(8):
  sids = sorted(E.query('entity==@ent')['snippet_id'].unique())
  out.append({'entity': ent, 'snippets': sids, 'texts': [snippets[i] for i in sids]})
pd.DataFrame(out)

,entity,snippets,texts
0,domain:acme.example,"[0, 1, 2]","[Support: contact helpdesk@acme.example, or vi..."
1,ip:203.0.113.10,"[0, 3]","[Support: contact helpdesk@acme.example, or vi..."
2,domain:api.vendor.example,[3],"[Third-party vendor: alerts@vendor.example, AP..."
3,domain:blog.acme.example,[4],[Community forum: user @john_doe posts link ht...
4,domain:status.acme.example,[0],"[Support: contact helpdesk@acme.example, or vi..."
5,domain:login.acme.example,[2],[A leaked paste references admin@acme.example ...
6,email:admin@acme.example,[2],[A leaked paste references admin@acme.example ...
7,email:alerts@vendor.example,[3],"[Third-party vendor: alerts@vendor.example, AP..."
